# 1. Setup and Download Dataset
Make sure to add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the Colab Secrets panel (the key icon on the left).
Alternatively, you can manually upload your `kaggle.json` file.

In [ ]:
!pip install -q kaggle

import os
from google.colab import userdata

try:
  os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
  os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except:
  print("Make sure you set your Kaggle secrets in Colab.")

!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset --unzip -p /content/

# 2. Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split

BASE_DIR = '/content'
TRAIN_DIR = os.path.join(BASE_DIR, 'Training')
TEST_DIR = os.path.join(BASE_DIR, 'Testing')

def get_class_paths(path):
  classes = []
  class_paths = []
  for label in os.listdir(path):
    label_path = os.path.join(path, label)
    if os.path.isdir(label_path):
      for image in os.listdir(label_path):
        image_path = os.path.join(label_path, image)
        classes.append(label)
        class_paths.append(image_path)
  return pd.DataFrame({'Class Path': class_paths, 'Class': classes})

tr_df = get_class_paths(TRAIN_DIR)
ts_df = get_class_paths(TEST_DIR)
valid_df, ts_df = train_test_split(ts_df, test_size=0.5, stratify = ts_df['Class'])

# 3. Train Xception Model

In [ ]:
batch_size = 32
img_size = (299, 299)
image_generator = ImageDataGenerator(
    rescale=1/255, rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
    horizontal_flip=True, brightness_range=(0.8, 1.2), zoom_range=0.2,
    shear_range=0.2, channel_shift_range=20.0, fill_mode='nearest')

tr_gen = image_generator.flow_from_dataframe(tr_df, x_col='Class Path', y_col='Class', batch_size=batch_size, target_size=img_size)
valid_gen = image_generator.flow_from_dataframe(valid_df, x_col='Class Path', y_col='Class', batch_size=batch_size, target_size=img_size)

img_shape = (299, 299, 3)
base_model = tf.keras.applications.Xception(include_top = False, weights = 'imagenet', input_shape = img_shape, pooling= 'max')
model = Sequential([
    base_model, Flatten(), Dropout(rate=0.3), Dense(256, activation='relu'), Dropout(rate=0.25), Dense(4, activation='softmax')
])
model.compile(Adamax(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy', Precision(), Recall()])

callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('xception_model.weights.h5', monitor='val_accuracy', save_best_only=True, save_weights_only=True, verbose=1)
]

print("Training Xception Model...")
hist = model.fit(tr_gen, epochs=20, validation_data=valid_gen, callbacks=callbacks)
model.save_weights("xception_model.weights.h5")
print("Saved Xception model weights!")

# 4. Train Custom CNN Model

In [ ]:
from tensorflow.keras import regularizers

batch_size = 16
img_size = (224, 224)
tr_gen = image_generator.flow_from_dataframe(tr_df, x_col='Class Path', y_col='Class', batch_size=batch_size, target_size=img_size)
valid_gen = image_generator.flow_from_dataframe(valid_df, x_col='Class Path', y_col='Class', batch_size=batch_size, target_size=img_size)

cnn_model = Sequential()
cnn_model.add(Conv2D(512, (3, 3), padding='same', input_shape=(224,224,3), activation='relu'))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Conv2D (256, (3, 3), padding='same', activation="relu"))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Dropout (0.25))
cnn_model.add(Conv2D (128, (3, 3), padding='same', activation='relu'))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Dropout (0.25))
cnn_model.add(Conv2D (64, (3, 3), padding='same', activation="relu"))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))
cnn_model.add(Flatten())
cnn_model.add(Dense (256, activation='relu', kernel_regularizer=regularizers.l2(0.01)))
cnn_model.add(Dropout (0.35))
cnn_model.add(Dense (4, activation='softmax'))

cnn_model.compile(Adamax(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy', Precision(), Recall()])

cnn_callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('cnn_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("Training CNN Model...")
history = cnn_model.fit(tr_gen, epochs=20, validation_data=valid_gen, callbacks=cnn_callbacks)
cnn_model.save("cnn_model.h5")
print("Saved CNN model!")

# 5. Download Models to Local Machine

In [ ]:
from google.colab import files
print("Downloading models to your computer...")
try:
  files.download('xception_model.weights.h5')
  files.download('cnn_model.h5')
except:
  print("Files not found. Make sure training completed.")

In [ ]:
# ==========================================
# Model Evaluation & ROC Curves
# ==========================================
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle
import numpy as np

# Generate predictions for the test set
# We'll use the CNN model for this evaluation
print("Generating predictions on the test set...")
ts_gen.reset()
preds = cnn_model.predict(ts_gen, verbose=1)

# Get true labels
y_true = ts_gen.classes
# Binarize the output for multi-class ROC
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
n_classes = y_true_bin.shape[1]

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], preds[:len(y_true), i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot all ROC curves
plt.figure(figsize=(10, 8))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'green'])
class_names = list(ts_gen.class_indices.keys())

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of {0} (area = {1:0.2f})'
             ''.format(class_names[i], roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()